In [1]:
import nndet_onnx

In [8]:
from nndet_onnx.nndet_onnx_inference_sw import (
    create_session,
    compute_anchors,
    process_single_image
)
import json 
from pathlib import Path

In [9]:
def run_mvpdet_programmatically():
    # 1. Configuration des chemins
    model_path = "path/to/model_onnx.onnx"
    plan_path = "path/to/plan_inference.json"
    image_path = "path/to/votre_image.nii.gz"
    output_dir = Path("path/to/output")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # 2. Charger le plan json
    with open(plan_path, "r") as f:
        plan_inference = json.load(f)
    
    patch_size = tuple(plan_inference["patch_size"])
    iou_threshold = plan_inference["inference_plan"].get("model_iou", 0.1)
    
    # 3. Créer la session ONNX (backends possibles : "cpu", "cuda", "openvino", "trt")
    session = create_session(model_path, backend="cpu", trt_fp16=False)
    batch_size = session.get_inputs()[0].shape[0]
    
    # 4. Pré-calculer les anchors (une seule fois pour toutes vos images)
    anchors_batch = compute_anchors(plan_inference, patch_size, batch_size)
    
    # 5. Analyser une image avec le pipeline complet
    nb_detections = process_single_image(
        session=session,
        image_path=image_path,
        output_dir=output_dir,
        plan_inference=plan_inference,
        patch_size=patch_size,
        batch_size=batch_size,
        anchors_batch=anchors_batch,
        iou_threshold=iou_threshold,
        overlap=0.5,
        score_thresh=0.5,
        min_size_mm=2.0,
        nms_backend="numpy", # ou "nndet"
        no_global_nms=False,
        export_pkl=False
    )
    
    print(f"Trouvé {nb_detections} objet(s) dans {image_path}!")